# Example research session

A short demonstration of the `quantkit` toolkit. This is a *template* — copy it,
change the tickers/questions, and explore. The teaching lives in the README; this
is just the tools in action.

In [ ]:
import sys; sys.path.append("..")   # import quantkit from the notebooks/ folder
import quantkit as qk
import matplotlib.pyplot as plt

prices = qk.load("prices")          # whatever build_dataset.py produced
prices.tail()

## Risk / return snapshot

In [ ]:
qk.summarize(prices).sort_values("Sharpe", ascending=False).round(2)

## A moving-average crossover on one stock vs buy-and-hold

In [ ]:
bt = qk.ma_crossover(prices["WES.AX"], short=50, long=200, cost=0.001)
print("Strategy Sharpe:", round(qk.sharpe(bt["strat_ret"]), 2))
print("Buy & Hold Sharpe:", round(qk.sharpe(bt["bh_ret"]), 2))

## Momentum, tested out-of-sample

In [ ]:
SPLIT = "2021-01-01"
monthly = prices.resample("ME").last()
train, _ = qk.train_test_split(monthly, SPLIT)

# pick the best settings on TRAIN only
best = qk.rank_momentum_params(train, lookbacks=[3, 6, 9, 12], top_ks=[2, 3, 4]).iloc[0]
lb, k = int(best.lookback), int(best.top_k)

# run those settings on the full series, score only the unseen TEST slice
strat = qk.momentum_portfolio(monthly, lb, k)
bench = qk.equal_weight(monthly)
test_strat = strat[strat.index >= SPLIT]
test_bench = bench[bench.index >= SPLIT]

print(f"Best train params: lookback={lb}, top_k={k}")
print("TRAIN strategy Sharpe :", round(qk.sharpe(qk.momentum_portfolio(train, lb, k), periods=12), 2))
print("TEST  strategy Sharpe :", round(qk.sharpe(test_strat, periods=12), 2))
print("TEST  benchmark Sharpe:", round(qk.sharpe(test_bench, periods=12), 2))

The gap between **train** and **test** Sharpe is the cost of overfitting,
made visible. A robust edge keeps a small gap *and* beats the benchmark
out-of-sample. See the README for the full reasoning and findings.